## Exploratory Data Analysis and Cleanup for Children's Books

In [64]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

In [65]:
DetectorFactory.seed = 42

In [66]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

True

In [67]:
#print(children_path)

In [68]:
comic_df = pd.read_json('../data/books/goodreads_books_mystery_thriller_crime.json', lines=True)

In [69]:
comic_df.shape

(219235, 29)

In [70]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,184737297X,15,[169353],US,,"[{'count': '159', 'name': 'to-read'}, {'count'...",,false,3.93,B007YLTG5I,"[439108, 522621, 116770, 1275927, 6202655, 840...","London, 1196. At the command of Richard the Li...",Hardcover,https://www.goodreads.com/book/show/6066814-cr...,"[{'author_id': '37778', 'role': ''}]",Simon & Schuster UK,400,6,9781847372970,4,,2009,https://www.goodreads.com/book/show/6066814-cr...,https://images.gr-assets.com/books/1328724803m...,6066814,186,6243149,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)"
1,,60,[1052227],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,true,4.33,B01NCIKAQX,[],,,https://www.goodreads.com/book/show/33394837-t...,"[{'author_id': '242185', 'role': ''}]",,318,,,,,,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269,54143148,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
2,,23,[953679],US,eng,"[{'count': '90', 'name': 'to-read'}, {'count':...",B01ALOWJN0,true,3.49,B01ALOWJN0,[],"BATHS, BANKS AND ROMAN INSURRECTION\nDetective...",,https://www.goodreads.com/book/show/29074697-t...,"[{'author_id': '15104629', 'role': ''}, {'auth...",Amazon Digital Services,,,,,,,https://www.goodreads.com/book/show/29074697-t...,https://s.gr-assets.com/assets/nophoto/book/11...,29074697,192,49305010,The Slaughtered Virgin of Zenopolis (Inspector...,The Slaughtered Virgin of Zenopolis (Inspector...
3,0854563903,8,[408775],US,,"[{'count': '51', 'name': 'to-read'}, {'count':...",,false,3.30,,[],"Gerald breezily introduced his wife, Helen, to...",Hardcover,https://www.goodreads.com/book/show/1902202.De...,"[{'author_id': '190988', 'role': ''}]",Ulverscroft,,1,9780854563906,12,Large Print,1975,https://www.goodreads.com/book/show/1902202.De...,https://s.gr-assets.com/assets/nophoto/book/11...,1902202,52,1903897,"Dead in the Morning (Patrick Grant, #1)","Dead in the Morning (Patrick Grant, #1)"
4,8838920931,3,[274410],US,ita,"[{'count': '48', 'name': 'to-read'}, {'count':...",,false,3.54,,[],"""I misteri di Eleusi"" e il quinto romanzo di A...",Paperback,https://www.goodreads.com/book/show/9671977-ar...,"[{'author_id': '337108', 'role': ''}, {'author...",Sellerio,659,,9788838920936,,,2006,https://www.goodreads.com/book/show/9671977-ar...,https://images.gr-assets.com/books/1474788304m...,9671977,22,2152906,Aristotele e i misteri di Eleusi,Aristotele e i misteri di Eleusi


In [71]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [72]:
df.shape

(219235, 29)

In [73]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [74]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [75]:
df.shape

(219235, 26)

In [76]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)",[169353],US,,184737297X,,B007YLTG5I,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",false,Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840..."
1,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2),[1052227],US,eng,,B01NCIKAQX,B01NCIKAQX,,54143148,33394837,60,4.33,269,"[{'count': '54', 'name': 'currently-reading'},...",true,,318,,,,,,,"[{'author_id': '242185', 'role': ''}]",[]
2,The Slaughtered Virgin of Zenopolis (Inspector...,The Slaughtered Virgin of Zenopolis (Inspector...,[953679],US,eng,,B01ALOWJN0,B01ALOWJN0,,49305010,29074697,23,3.49,192,"[{'count': '90', 'name': 'to-read'}, {'count':...",true,,,Amazon Digital Services,,,,,"BATHS, BANKS AND ROMAN INSURRECTION\nDetective...","[{'author_id': '15104629', 'role': ''}, {'auth...",[]
3,"Dead in the Morning (Patrick Grant, #1)","Dead in the Morning (Patrick Grant, #1)",[408775],US,,0854563903,,,9780854563906,1903897,1902202,8,3.30,52,"[{'count': '51', 'name': 'to-read'}, {'count':...",false,Hardcover,,Ulverscroft,1,12,1975,Large Print,"Gerald breezily introduced his wife, Helen, to...","[{'author_id': '190988', 'role': ''}]",[]
4,Aristotele e i misteri di Eleusi,Aristotele e i misteri di Eleusi,[274410],US,ita,8838920931,,,9788838920936,2152906,9671977,3,3.54,22,"[{'count': '48', 'name': 'to-read'}, {'count':...",false,Paperback,659,Sellerio,,,2006,,"""I misteri di Eleusi"" e il quinto romanzo di A...","[{'author_id': '337108', 'role': ''}, {'author...",[]


In [77]:
df['clean_title'] = df['title_without_series']

In [78]:
df.shape

(219235, 27)

### Detecting languages

In [79]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [80]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 82451/82451 [03:23<00:00, 405.31it/s]


In [81]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)",[169353],US,,184737297X,,B007YLTG5I,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",false,Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840...","Crowner Royal (Crowner John Mystery, #13)",eng,"Crowner Royal (Crowner John Mystery, #13) Lond...",eng
1,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2),[1052227],US,eng,,B01NCIKAQX,B01NCIKAQX,,54143148,33394837,60,4.33,269,"[{'count': '54', 'name': 'currently-reading'},...",true,,318,,,,,,,"[{'author_id': '242185', 'role': ''}]",[],The House of Memory (Pluto's Snitch #2),eng,The House of Memory (Pluto's Snitch #2),nan
2,The Slaughtered Virgin of Zenopolis (Inspector...,The Slaughtered Virgin of Zenopolis (Inspector...,[953679],US,eng,,B01ALOWJN0,B01ALOWJN0,,49305010,29074697,23,3.49,192,"[{'count': '90', 'name': 'to-read'}, {'count':...",true,,,Amazon Digital Services,,,,,"BATHS, BANKS AND ROMAN INSURRECTION\nDetective...","[{'author_id': '15104629', 'role': ''}, {'auth...",[],The Slaughtered Virgin of Zenopolis (Inspector...,eng,The Slaughtered Virgin of Zenopolis (Inspector...,nan
3,"Dead in the Morning (Patrick Grant, #1)","Dead in the Morning (Patrick Grant, #1)",[408775],US,,0854563903,,,9780854563906,1903897,1902202,8,3.30,52,"[{'count': '51', 'name': 'to-read'}, {'count':...",false,Hardcover,,Ulverscroft,1,12,1975,Large Print,"Gerald breezily introduced his wife, Helen, to...","[{'author_id': '190988', 'role': ''}]",[],"Dead in the Morning (Patrick Grant, #1)",eng,"Dead in the Morning (Patrick Grant, #1) Gerald...",eng
4,Aristotele e i misteri di Eleusi,Aristotele e i misteri di Eleusi,[274410],US,ita,8838920931,,,9788838920936,2152906,9671977,3,3.54,22,"[{'count': '48', 'name': 'to-read'}, {'count':...",false,Paperback,659,Sellerio,,,2006,,"""I misteri di Eleusi"" e il quinto romanzo di A...","[{'author_id': '337108', 'role': ''}, {'author...",[],Aristotele e i misteri di Eleusi,ita,"Aristotele e i misteri di Eleusi ""I misteri di...",nan


In [82]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [83]:
df.shape

(219235, 30)

In [84]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [85]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,"Crowner Royal (Crowner John Mystery, #13)",[169353],eng,184737297X,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840..."
1,The House of Memory (Pluto's Snitch #2),[1052227],eng,,,54143148,33394837,60,4.33,269,"[{'count': '54', 'name': 'currently-reading'},...",,318,,,,,,,"[{'author_id': '242185', 'role': ''}]",[]
2,The Slaughtered Virgin of Zenopolis (Inspector...,[953679],eng,,,49305010,29074697,23,3.49,192,"[{'count': '90', 'name': 'to-read'}, {'count':...",,,Amazon Digital Services,,,,,"BATHS, BANKS AND ROMAN INSURRECTION\nDetective...","[{'author_id': '15104629', 'role': ''}, {'auth...",[]
3,"Dead in the Morning (Patrick Grant, #1)",[408775],eng,0854563903,9780854563906,1903897,1902202,8,3.30,52,"[{'count': '51', 'name': 'to-read'}, {'count':...",Hardcover,,Ulverscroft,1,12,1975,Large Print,"Gerald breezily introduced his wife, Helen, to...","[{'author_id': '190988', 'role': ''}]",[]
4,Aristotele e i misteri di Eleusi,[274410],ita,8838920931,9788838920936,2152906,9671977,3,3.54,22,"[{'count': '48', 'name': 'to-read'}, {'count':...",Paperback,659,Sellerio,,,2006,,"""I misteri di Eleusi"" e il quinto romanzo di A...","[{'author_id': '337108', 'role': ''}, {'author...",[]


In [86]:
df.shape

(219235, 21)

In [87]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [88]:
df.shape

(173332, 21)

In [89]:
df['format'].value_counts().shape

(207,)

In [90]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['Hardcover', '', 'Audio', 'Paperback', 'Audiobook', 'Audio CD', 'ebook', 'Audible Audio', 'Kindle Edition', 'Mass Market Paperback', 'Unknown Binding', 'Library Binding', 'Audio Cassette', 'CD-ROM', 'Leather Bound', 'MP3 CD', 'Trade Paperback', 'mass market paperback', 'Proof', 'ebook &amp; paperback', 'Sony Reader', 'Paperback - Large Print', 'paper', 'Leatherbound', 'Soft Cover', 'trade paperback', 'HARDCOVER', 'webtoon', 'hardback', 'Tradepaperback', 'Unbound', 'paperback', 'Nook', 'Mass Market Paperback - Reissue', 'softcover', 'Board Book', 'Radio drama', 'Kindle, Paperback', 'large print', 'Large Print', 'ebook&amp;paperback', 'Paperback and ebook', 'ebook, paperback', 'Softcover', 'MP3 Audio', 'Digital Audio', 'Playaway', 'true first ed HC, re-bound by library', 'audio', 'hardcover', 'Graphic Novel', 'ARC', 'Kindle Edition, Paperback', 'Paper', 'MP3 Book', 'Export Trade Paperback', 'Hardback', 'Online Freebie', 'Audio Cassette - Abridged', 'Fictional Novel', '10 Compact Discs',

In [91]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard print
    "Hardcover",
    "HARDCOVER",
    "hardcover",
    "Hardback",
    "hardback",
    "hard cover",
    "Hard cover",
    "Hardcover (Large Print)",
    "Hardcover - Large Print",
    "Large Print Hardcover",
    "Limited Edition Hardcover",
    "Hardcover, Book Club Edition",
    "Faux leather hardcover",
    "true first ed HC, re-bound by library",
    "Hardcover-LP",
    "Hardover",
    "Harcover",

    "Paperback",
    "paperback",
    "(Paperback)",
    "pb",
    "paper",
    "Paper",
    "Bookpaper",
    "Soft Cover",
    "Soft cover",
    "softcover",
    "Softcover",
    "Mass Market Paperback",
    "mass market paperback",
    "mass market Paperback",
    "Mass-market paperback",
    "Mass-Market",
    "Mass Market Paperback - Reissue",
    "Trade Paperback",
    "trade paperback",
    "Trade paperback",
    "Tradepaperback",
    "Trade Paper back",
    "Trade Paperbck",
    "Export Trade Paperback",
    "Open Market Paperback",
    "Paperback - C Format",
    "Paperback - Large Print",
    "Large Print Paperback",
    "large print",
    "Large Print",
    "UK paperback",
    "UK Paperback",
    "Pocket Paperback",
    "Pocket Book",
    "Perfect Paperback",
    "Paperback Manga",
    "Capa Mole",
    "Capa Dura",
    "Broche",

    # Bindings / print variants
    "Library Binding",
    "Library Edition",
    "Leather Bound",
    "Leatherbound",
    "Leatherette Bound",
    "Faux leather hardcover",
    "Textbook Binding",
    "Unbound",
    "cloth",

    # Board / comic / graphic / novelty
    "Board Book",
    "Board book",
    "Comic",
    "Comic Book",
    "Graphic Novel",
    "Novelty Book",
    "webtoon",

    # Proofs / ARCs
    "Proof",
    "ARC",
    "Uncorrected Proof",
    "Advance Reader's Copy",
    "Advance Review Copy",

    # Sets / omnibus
    "Box Set",
    "box set",
    "Boxed Set (paperback)",
    "Part-In-Omnibus",

    # Ebooks / digital readable text
    "ebook",
    "eBook",
    "epub",
    "Mobi",
    "PDB",
    "Kindle Edition",
    "Kindle",
    "kindle",
    "Kindle eBook",
    "Kindel Edition",
    "Sony Reader",
    "Nook",
    "Kobo",
    "Adobe Digital Edition",
    "iBook",

    # Mixed print + ebook
    "ebook &amp; paperback",
    "ebook&amp;paperback",
    "Paperback and ebook",
    "Paperback and eBook",
    "Paperback and EBook",
    "Paperback, ebook",
    "Paperback, eBook",
    "Paperback, e-Book",
    "Paperback and e-book",
    "ebook, paperback",
    "ebook and paperback",
    "Ebook and paperback",
    "Ebook &amp; Paperback",
    "E-book &amp; Paperback",
    "Paperback; Digital (epub, mobi)",
    "Paperback &amp; PDF",
    "Paperback/Kindle",
    "Paperback, Kindle",
    "Paperback and Kindle",
    "Paperback &amp; Kindle",
    "Kindle, Paperback",
    "Kindle and Paperback",
    "Kindle Edition, Paperback",
    "Kindle Edition and Paperback",
    "Kindle/Paperback",
    "ebook and print",
    "ebook, print",
    "Print and ebook",
    "print/Kindle",
    "electronic &amp; print",
    "ebook and trade paperback",
    "ebook, kindle",
    "Print and Download",

    # Online readable text
    "Online",
    "online fiction",
    "Original Online Fiction",
    "Free Online Fiction",
    "Online Fiction - Complete",
    "Online Publication",
    "Online Freebie",
    "Online post",
    "Web Serial Novel",
    "Internet Platform",

    # Other readable text containers
    "Book",
    "Fictional Novel",
    "Chapbook",
    "Digital Chapbook",
    "Periodical",
    "Print",
}

# 2. Define formats to drop
drop_formats = {
    # Missing / unknown
    "",
    "Unknown Binding",
    "Unknown",
    "Abriged Unknown Binding",
    "Multiple Formats",

    # Audio
    "Audio",
    "audio",
    "Audiobook",
    "audiobook",
    "Audible Audio",
    "audible audio",
    "Audio Book",
    "audio cassette",
    "Audio Cassette",
    "Audio Cassette - Abridged",
    "Audio Cassette - Unabridged",
    "Audio Cassette (Libr.) (Unabr.)",
    "Audiocassette",
    "audiocassette",
    "Audiocassette (unabridged)",
    "Audio CD",
    "audio CD",
    "Audio cd",
    "Audio CD (Unabr.)",
    "Abriged Audio CD",
    "Unabriged Audio CD",
    "MP3 CD",
    "MP3 Audio",
    "MP3 Book",
    "MP3",
    "MP3 Audible",
    "MP3 Audio Download",
    "MP3 Download",
    "mp3 Audiobook",
    "Audible MP3 Download",
    "Digital Audio",
    "Digital Audio (Unabridged)",
    "eAudiobook",
    "Audiobook - Audible Download",
    "Audiobook Unabridged",
    "Playaway",
    "Playaway Audio",
    "Audio Playaway",
    "Preloaded Digital Audio Player",
    "Preloaded Digital Player",
    "Podcast",
    "Podiobook",
    "podcast/podiobooks",
    "Podcast / Audiobook",
    "Radio drama",

    # Mixed but contains audio; drop if you want no audio for now
    "ebook, paperback, Audible audio",
    "Paperback, eBook, Audible",

    # Video / discs / old media
    "CD-ROM",
    "CD",
    "VHS Tape",
    "Diskette",
    "Vook",

    # Not useful as book format / likely bad metadata
    "Seeds Of Civilization",
    "MIRA",
    "Crime and Investigation",
    "Mystery",
}


# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      112336
other      60993
review         3
Name: count, dtype: int64

In [92]:
df = df[df["format_group"] == "text"].copy()

In [93]:
df["format_group"].value_counts()

format_group
text    112336
Name: count, dtype: int64

In [94]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
0,"Crowner Royal (Crowner John Mystery, #13)",[169353],eng,184737297X,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840...",Hardcover,text
3,"Dead in the Morning (Patrick Grant, #1)",[408775],eng,0854563903,9780854563906,1903897,1902202,8,3.30,52,"[{'count': '51', 'name': 'to-read'}, {'count':...",Hardcover,,Ulverscroft,1,12,1975,Large Print,"Gerald breezily introduced his wife, Helen, to...","[{'author_id': '190988', 'role': ''}]",[],Hardcover,text
7,Wycliffe and the Cycle of Death,[326237],eng,0752844458,9780752844459,2831381,2805495,8,3.61,58,"[{'count': '38', 'name': 'to-read'}, {'count':...",Paperback,320,Orion,2,8,2001,,A respectable bookseller is found bludgeoned a...,"[{'author_id': '1533165', 'role': ''}]",[],Paperback,text
8,The Cost of Doing Business,[],eng,8293326247,9788293326243,42251489,22722787,6,4.14,18,"[{'count': '171', 'name': 'to-read'}, {'count'...",Paperback,228,280 Steps,4,11,2014,,"""Poetic, down trodden and nihilistic, Jonathan...","[{'author_id': '4577517', 'role': ''}]",[],Paperback,text
11,"Polar Star (Arkady Renko, #2)",[163090],eng,0345367650,9780345367655,2640521,778285,224,3.99,5376,"[{'count': '965', 'name': 'to-read'}, {'count'...",Paperback,366,Ballantine Books,13,6,1990,,"In the long-awaited sequel to Gorky Park, Arka...","[{'author_id': '8258', 'role': ''}]","[312134, 12160902, 8518058, 209199, 171622, 92...",Paperback,text


In [95]:
df.shape

(112336, 23)

In [96]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [97]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [98]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
0,"Crowner Royal (Crowner John Mystery, #13)",[169353],eng,184737297X,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",Hardcover,400,Simon & Schuster UK,2009,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840..."
3,"Dead in the Morning (Patrick Grant, #1)",[408775],eng,0854563903,9780854563906,1903897,1902202,8,3.30,52,"[{'count': '51', 'name': 'to-read'}, {'count':...",Hardcover,,Ulverscroft,1975,"Gerald breezily introduced his wife, Helen, to...","[{'author_id': '190988', 'role': ''}]",[]
7,Wycliffe and the Cycle of Death,[326237],eng,0752844458,9780752844459,2831381,2805495,8,3.61,58,"[{'count': '38', 'name': 'to-read'}, {'count':...",Paperback,320,Orion,2001,A respectable bookseller is found bludgeoned a...,"[{'author_id': '1533165', 'role': ''}]",[]
8,The Cost of Doing Business,[],eng,8293326247,9788293326243,42251489,22722787,6,4.14,18,"[{'count': '171', 'name': 'to-read'}, {'count'...",Paperback,228,280 Steps,2014,"""Poetic, down trodden and nihilistic, Jonathan...","[{'author_id': '4577517', 'role': ''}]",[]
11,"Polar Star (Arkady Renko, #2)",[163090],eng,0345367650,9780345367655,2640521,778285,224,3.99,5376,"[{'count': '965', 'name': 'to-read'}, {'count'...",Paperback,366,Ballantine Books,1990,"In the long-awaited sequel to Gorky Park, Arka...","[{'author_id': '8258', 'role': ''}]","[312134, 12160902, 8518058, 209199, 171622, 92..."


In [99]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [100]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [101]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [102]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [103]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,137,Chandler: Stories and Early Novels,"[{'author_id': '1377', 'role': ''}, {'author_i...",[[1035490]],eng,[1883011078],[9781883011079],[2046],[Library of America],[1995.0],The library of America is dedicated to publish...,"[{'count': '605', 'name': 'to-read'}, {'count'...","[[29997, 166915, 12230129, 50186, 113090, 1230...",56,1176,4.49,1199.0
1,434,The Black Dahlia,"[{'author_id': '2887', 'role': ''}]",[[230961]],eng,"[0446618128, 0099366517, 0099537869, 044540525...","[9780446618120, 9780099366515, 9780099537861, ...","[15741396, 85587, 895583, 11783478, 882524, 54...","[Arrow Books, Grand Central Publishing, Arrow,...","[1993.0, 2006.0, 1994.0, 2011.0, 1988.0, 1987....",What's to Love: This gripping graphic novel ad...,"[{'count': '3471', 'name': 'to-read'}, {'count...","[[19161909, 776159, 280727, 712372, 867622, 28...",1553,62702,3.75,397.0
2,995,Derailed,"[{'author_id': '57057', 'role': ''}]",[],eng,"[0446514152, 044661372X, 0446696692]","[9780446514156, 9780446613729, 9780446696692]","[5952238, 1387774, 314361]",[Grand Central Publishing],"[2007.0, 2004.0, 2005.0]",Two Strangers on a Train. A Taste of Temptatio...,"[{'count': '668', 'name': 'to-read'}, {'count'...","[[130391, 596239, 1024724, 79727, 6431264, 285...",10,37,3.77,418.0
3,1285,"Plum Boxed Set 2 (Stephanie Plum, #4-6)","[{'author_id': '2384', 'role': ''}]",[[183284]],eng,[0312947445],[9780312947446],[148140],[St. Martin's Paperbacks],[2007.0],This boxed set contains three hilarious Stepha...,"[{'count': '371', 'name': 'to-read'}, {'count'...","[[287573, 12961168, 2137350, 1376643, 2860451,...",30,1280,4.43,NaN
4,1616,"The High Window (Philip Marlowe, #3)","[{'author_id': '1377', 'role': ''}]",[[855278]],eng,"[0140008519, 0241956293, 0140108939, 039475826...","[9780140008517, 9780241956298, 9780140108934, ...","[1930777, 25638586, 246991, 11373619, 2049, 74...","[Penguin, Ballantine Books, Penguin Books, Vin...","[1951.0, 1973.0, 2011.0, 1988.0, 2002.0, 1976.0]","Raymond Chandler is a master."" --""The New York...","[{'count': '4610', 'name': 'to-read'}, {'count...","[[30003, 867622, 280727, 844793, 19161909]]",439,11370,4.08,272.0


In [104]:
df = books_work_df.copy()

In [105]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,137,Chandler: Stories and Early Novels,"[{'author_id': '1377', 'role': ''}, {'author_i...",[[1035490]],eng,[1883011078],[9781883011079],[2046],[Library of America],[1995.0],The library of America is dedicated to publish...,"[{'count': '605', 'name': 'to-read'}, {'count'...","[[29997, 166915, 12230129, 50186, 113090, 1230...",56,1176,4.49,1199.0
1,434,The Black Dahlia,"[{'author_id': '2887', 'role': ''}]",[[230961]],eng,"[0446618128, 0099366517, 0099537869, 044540525...","[9780446618120, 9780099366515, 9780099537861, ...","[15741396, 85587, 895583, 11783478, 882524, 54...","[Arrow Books, Grand Central Publishing, Arrow,...","[1993.0, 2006.0, 1994.0, 2011.0, 1988.0, 1987....",What's to Love: This gripping graphic novel ad...,"[{'count': '3471', 'name': 'to-read'}, {'count...","[[19161909, 776159, 280727, 712372, 867622, 28...",1553,62702,3.75,397.0
2,995,Derailed,"[{'author_id': '57057', 'role': ''}]",[],eng,"[0446514152, 044661372X, 0446696692]","[9780446514156, 9780446613729, 9780446696692]","[5952238, 1387774, 314361]",[Grand Central Publishing],"[2007.0, 2004.0, 2005.0]",Two Strangers on a Train. A Taste of Temptatio...,"[{'count': '668', 'name': 'to-read'}, {'count'...","[[130391, 596239, 1024724, 79727, 6431264, 285...",10,37,3.77,418.0
3,1285,"Plum Boxed Set 2 (Stephanie Plum, #4-6)","[{'author_id': '2384', 'role': ''}]",[[183284]],eng,[0312947445],[9780312947446],[148140],[St. Martin's Paperbacks],[2007.0],This boxed set contains three hilarious Stepha...,"[{'count': '371', 'name': 'to-read'}, {'count'...","[[287573, 12961168, 2137350, 1376643, 2860451,...",30,1280,4.43,NaN
4,1616,"The High Window (Philip Marlowe, #3)","[{'author_id': '1377', 'role': ''}]",[[855278]],eng,"[0140008519, 0241956293, 0140108939, 039475826...","[9780140008517, 9780241956298, 9780140108934, ...","[1930777, 25638586, 246991, 11373619, 2049, 74...","[Penguin, Ballantine Books, Penguin Books, Vin...","[1951.0, 1973.0, 2011.0, 1988.0, 2002.0, 1976.0]","Raymond Chandler is a master."" --""The New York...","[{'count': '4610', 'name': 'to-read'}, {'count...","[[30003, 867622, 280727, 844793, 19161909]]",439,11370,4.08,272.0


In [106]:
df.shape

(65486, 17)

In [107]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description              2479
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                5963
dtype: int64

In [108]:
df.shape

(65486, 17)

In [109]:
df = df[df["description"].notna()].copy()

In [110]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description                 0
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                5388
dtype: int64

In [111]:
df.shape

(63007, 17)

In [112]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,137,Chandler: Stories and Early Novels,"[{'author_id': '1377', 'role': ''}, {'author_i...",[[1035490]],eng,[1883011078],[9781883011079],[2046],[Library of America],[1995.0],The library of America is dedicated to publish...,"[{'count': '605', 'name': 'to-read'}, {'count'...","[[29997, 166915, 12230129, 50186, 113090, 1230...",56,1176,4.49,1199.0
1,434,The Black Dahlia,"[{'author_id': '2887', 'role': ''}]",[[230961]],eng,"[0446618128, 0099366517, 0099537869, 044540525...","[9780446618120, 9780099366515, 9780099537861, ...","[15741396, 85587, 895583, 11783478, 882524, 54...","[Arrow Books, Grand Central Publishing, Arrow,...","[1993.0, 2006.0, 1994.0, 2011.0, 1988.0, 1987....",What's to Love: This gripping graphic novel ad...,"[{'count': '3471', 'name': 'to-read'}, {'count...","[[19161909, 776159, 280727, 712372, 867622, 28...",1553,62702,3.75,397.0
2,995,Derailed,"[{'author_id': '57057', 'role': ''}]",[],eng,"[0446514152, 044661372X, 0446696692]","[9780446514156, 9780446613729, 9780446696692]","[5952238, 1387774, 314361]",[Grand Central Publishing],"[2007.0, 2004.0, 2005.0]",Two Strangers on a Train. A Taste of Temptatio...,"[{'count': '668', 'name': 'to-read'}, {'count'...","[[130391, 596239, 1024724, 79727, 6431264, 285...",10,37,3.77,418.0
3,1285,"Plum Boxed Set 2 (Stephanie Plum, #4-6)","[{'author_id': '2384', 'role': ''}]",[[183284]],eng,[0312947445],[9780312947446],[148140],[St. Martin's Paperbacks],[2007.0],This boxed set contains three hilarious Stepha...,"[{'count': '371', 'name': 'to-read'}, {'count'...","[[287573, 12961168, 2137350, 1376643, 2860451,...",30,1280,4.43,NaN
4,1616,"The High Window (Philip Marlowe, #3)","[{'author_id': '1377', 'role': ''}]",[[855278]],eng,"[0140008519, 0241956293, 0140108939, 039475826...","[9780140008517, 9780241956298, 9780140108934, ...","[1930777, 25638586, 246991, 11373619, 2049, 74...","[Penguin, Ballantine Books, Penguin Books, Vin...","[1951.0, 1973.0, 2011.0, 1988.0, 2002.0, 1976.0]","Raymond Chandler is a master."" --""The New York...","[{'count': '4610', 'name': 'to-read'}, {'count...","[[30003, 867622, 280727, 844793, 19161909]]",439,11370,4.08,272.0


In [113]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [114]:
df.shape

(63007, 15)

In [115]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,137,Chandler: Stories and Early Novels,"[{'author_id': '1377', 'role': ''}, {'author_i...",[[1035490]],[1883011078],[9781883011079],[2046],[Library of America],The library of America is dedicated to publish...,"[{'count': '605', 'name': 'to-read'}, {'count'...","[[29997, 166915, 12230129, 50186, 113090, 1230...",56,1176,4.49,1199.0
1,434,The Black Dahlia,"[{'author_id': '2887', 'role': ''}]",[[230961]],"[0446618128, 0099366517, 0099537869, 044540525...","[9780446618120, 9780099366515, 9780099537861, ...","[15741396, 85587, 895583, 11783478, 882524, 54...","[Arrow Books, Grand Central Publishing, Arrow,...",What's to Love: This gripping graphic novel ad...,"[{'count': '3471', 'name': 'to-read'}, {'count...","[[19161909, 776159, 280727, 712372, 867622, 28...",1553,62702,3.75,397.0
2,995,Derailed,"[{'author_id': '57057', 'role': ''}]",[],"[0446514152, 044661372X, 0446696692]","[9780446514156, 9780446613729, 9780446696692]","[5952238, 1387774, 314361]",[Grand Central Publishing],Two Strangers on a Train. A Taste of Temptatio...,"[{'count': '668', 'name': 'to-read'}, {'count'...","[[130391, 596239, 1024724, 79727, 6431264, 285...",10,37,3.77,418.0
3,1285,"Plum Boxed Set 2 (Stephanie Plum, #4-6)","[{'author_id': '2384', 'role': ''}]",[[183284]],[0312947445],[9780312947446],[148140],[St. Martin's Paperbacks],This boxed set contains three hilarious Stepha...,"[{'count': '371', 'name': 'to-read'}, {'count'...","[[287573, 12961168, 2137350, 1376643, 2860451,...",30,1280,4.43,NaN
4,1616,"The High Window (Philip Marlowe, #3)","[{'author_id': '1377', 'role': ''}]",[[855278]],"[0140008519, 0241956293, 0140108939, 039475826...","[9780140008517, 9780241956298, 9780140108934, ...","[1930777, 25638586, 246991, 11373619, 2049, 74...","[Penguin, Ballantine Books, Penguin Books, Vin...","Raymond Chandler is a master."" --""The New York...","[{'count': '4610', 'name': 'to-read'}, {'count...","[[30003, 867622, 280727, 844793, 19161909]]",439,11370,4.08,272.0


In [116]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [117]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [118]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [119]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [120]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [121]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [122]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '605', 'name': 'to-read'}, {'count': '39', 'name': 'currently-reading'}, {'count': '29', 'name': 'fiction'}, {'count': '27', 'name': 'mystery'}, {'count': '16', 'name': 'library-of-america'}, {'count': '16', 'name': 'crime'}, {'count': '15', 'name': 'short-stories'}, {'count': '12', 'name': 'favorites'}, {'count': '10', 'name': 'owned'}, {'count': '9', 'name': 'noir'}, {'count': '7', 'name': 'literature'}, {'count': '6', 'name': 'mysteries'}, {'count': '6', 'name': 'detective'}, {'count': '6', 'name': 'classics'}, {'count': '5', 'name': 'hard-boiled'}, {'count': '4', 'name': 'mystery-thriller'}, {'count': '4', 'name': 'american'}, {'count': '4', 'name': 'novels'}, {'count': '3', 'name': 'my-library'}, {'count': '3', 'name': 'loa'}, {'count': '3', 'name': 'anthology'}, {'count': '2', 'name': 'short-story'}, {'count': '2', 'name': 'crime-fiction'}, {'count': '2', 'name': 'genre-fiction'}, {'count': '2', 'name': 'novel'}, {'count': '2', 'name': 'american-fiction'}, {'co

In [123]:
int_df = df[['popular_shelves']]

In [124]:
int_df.head()

,popular_shelves
0,"[{'count': '605', 'name': 'to-read'}, {'count'..."
1,"[{'count': '3471', 'name': 'to-read'}, {'count..."
2,"[{'count': '668', 'name': 'to-read'}, {'count'..."
3,"[{'count': '371', 'name': 'to-read'}, {'count'..."
4,"[{'count': '4610', 'name': 'to-read'}, {'count..."


In [125]:
int_df.shape

(63007, 1)

In [126]:
int_df.to_csv('../data/intermediate/mystery_shelves.csv', index=False)

In [127]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/mystery_tag_mapping.csv')

In [128]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,43906278,62533
1,currently-reading,status,drop,3059538,52006
2,mystery,mystery,keep,1980904,51878
3,fiction,fiction,keep,1076903,44414
4,thriller,thriller,keep,532664,30558
5,crime,crime,keep,424613,31148
6,favorites,review,drop,386582,23833
7,kindle,ownership,drop,312817,33770
8,series,noise,drop,291941,26188
9,owned,ownership,drop,284423,32693


In [129]:
vocab_df = shelves.copy()

In [130]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [131]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,43906278,62533
1,currently-reading,status,drop,3059538,52006
2,mystery,mystery,keep,1980904,51878
3,fiction,fiction,keep,1076903,44414
4,thriller,thriller,keep,532664,30558


In [132]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [133]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [134]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '605', 'name': 'to-read'}, {'count'...","[{'count': '29', 'name': 'fiction'}, {'count':..."
1,"[{'count': '3471', 'name': 'to-read'}, {'count...","[{'count': '735', 'name': 'mystery'}, {'count'..."
2,"[{'count': '668', 'name': 'to-read'}, {'count'...","[{'count': '48', 'name': 'thriller'}, {'count'..."
3,"[{'count': '371', 'name': 'to-read'}, {'count'...","[{'count': '27', 'name': 'mystery'}, {'count':..."
4,"[{'count': '4610', 'name': 'to-read'}, {'count...","[{'count': '364', 'name': 'mystery'}, {'count'..."


In [135]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '605', 'name': 'to-read'}, {'count': '39', 'name': 'currently-reading'}, {'count': '29', 'name': 'fiction'}, {'count': '27', 'name': 'mystery'}, {'count': '16', 'name': 'library-of-america'}, {'count': '16', 'name': 'crime'}, {'count': '15', 'name': 'short-stories'}, {'count': '12', 'name': 'favorites'}, {'count': '10', 'name': 'owned'}, {'count': '9', 'name': 'noir'}, {'count': '7', 'name': 'literature'}, {'count': '6', 'name': 'mysteries'}, {'count': '6', 'name': 'detective'}, {'count': '6', 'name': 'classics'}, {'count': '5', 'name': 'hard-boiled'}, {'count': '4', 'name': 'mystery-thriller'}, {'count': '4', 'name': 'american'}, {'count': '4', 'name': 'novels'}, {'count': '3', 'name': 'my-library'}, {'count': '3', 'name': 'loa'}, {'count': '3', 'name': 'anthology'}, {'count': '2', 'name': 'short-story'}, {'count': '2', 'name': 'crime-fiction'}, {'count': '2', 'name': 'genre-fiction'}, {'count': '2', 'name': 'novel'}, {'count': '2', 'name': 'americ

In [136]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [137]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [138]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '1088', 'name': 'mystery'}, {'count': '887', 'name': 'crime'}, {'count': '642', 'name': 'fiction'}, {'count': '321', 'name': 'noir'}, {'count': '298', 'name': 'historical-fiction'}, {'count': '290', 'name': 'thriller'}, {'count': '175', 'name': 'true-crime'}, {'count': '121', 'name': 'american-history'}, {'count': '75', 'name': 'nonfiction'}, {'count': '45', 'name': 'angels'}, {'count': '39', 'name': 'history'}, {'count': '53', 'name': 'adult'}, {'count': '38', 'name': 'horror'}, {'count': '53', 'name': 'classics'}, {'count': '26', 'name': '20th-century'}, {'count': '19', 'name': 'contemporary'}, {'count': '12', 'name': 'spy'}, {'count': '12', 'name': 'police-procedural'}]


In [139]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '205', 'name': 'fiction'}
{'count': '194', 'name': 'crime'}
{'count': '164', 'name': 'mystery'}
{'count': '63', 'name': 'thriller'}
{'count': '20', 'name': 'noir'}
{'count': '21', 'name': 'adventure'}
{'count': '7', 'name': 'italy'}
{'count': '7', 'name': 'classics'}
{'count': '6', 'name': 'humor'}
{'count': '9', 'name': 'american-history'}
{'count': '5', 'name': 'adult'}
{'count': '7', 'name': 'westerns'}
{'count': '3', 'name': '20th-century'}
{'count': '3', 'name': 'contemporary'}


In [140]:
df.shape

(63007, 18)

In [141]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,137,Chandler: Stories and Early Novels,"[{'author_id': '1377', 'role': ''}, {'author_i...",[1035490],[1883011078],[9781883011079],[2046],[Library of America],The library of America is dedicated to publish...,"[{'count': '605', 'name': 'to-read'}, {'count'...","[29997, 166915, 12230129, 50186, 113090, 12306...",56,1176,4.49,1199.0,"[1377, 71475]",0,"[{'count': '35', 'name': 'fiction'}, {'count':..."
1,434,The Black Dahlia,"[{'author_id': '2887', 'role': ''}]",[230961],"[0446618128, 0099366517, 0099537869, 044540525...","[9780446618120, 9780099366515, 9780099537861, ...","[15741396, 85587, 895583, 11783478, 882524, 54...","[Arrow Books, Grand Central Publishing, Arrow,...",What's to Love: This gripping graphic novel ad...,"[{'count': '3471', 'name': 'to-read'}, {'count...","[19161909, 776159, 280727, 712372, 867622, 288...",1553,62702,3.75,397.0,[2887],0,"[{'count': '1088', 'name': 'mystery'}, {'count..."
2,995,Derailed,"[{'author_id': '57057', 'role': ''}]",[],"[0446514152, 044661372X, 0446696692]","[9780446514156, 9780446613729, 9780446696692]","[5952238, 1387774, 314361]",[Grand Central Publishing],Two Strangers on a Train. A Taste of Temptatio...,"[{'count': '668', 'name': 'to-read'}, {'count'...","[130391, 596239, 1024724, 79727, 6431264, 2857...",10,37,3.77,418.0,[57057],0,"[{'count': '86', 'name': 'thriller'}, {'count'..."
3,1285,Plum Boxed Set 2,"[{'author_id': '2384', 'role': ''}]",[183284],[0312947445],[9780312947446],[148140],[St. Martin's Paperbacks],This boxed set contains three hilarious Stepha...,"[{'count': '371', 'name': 'to-read'}, {'count'...","[287573, 12961168, 2137350, 1376643, 2860451, ...",30,1280,4.43,NaN,[2384],1,"[{'count': '37', 'name': 'mystery'}, {'count':..."
4,1616,The High Window,"[{'author_id': '1377', 'role': ''}]",[855278],"[0140008519, 0241956293, 0140108939, 039475826...","[9780140008517, 9780241956298, 9780140108934, ...","[1930777, 25638586, 246991, 11373619, 2049, 74...","[Penguin, Ballantine Books, Penguin Books, Vin...","Raymond Chandler is a master."" --""The New York...","[{'count': '4610', 'name': 'to-read'}, {'count...","[30003, 867622, 280727, 844793, 19161909]",439,11370,4.08,272.0,[1377],0,"[{'count': '612', 'name': 'mystery'}, {'count'..."


In [142]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [143]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [144]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,137,Chandler: Stories and Early Novels,"[1377, 71475]",[1035490],[1883011078],[9781883011079],[2046],[Library of America],The library of America is dedicated to publish...,"[29997, 166915, 12230129, 50186, 113090, 12306...",56,1176,4.49,1199.0,0,"[{'count': '35', 'name': 'fiction'}, {'count':..."
1,434,The Black Dahlia,[2887],[230961],"[0446618128, 0099366517, 0099537869, 044540525...","[9780446618120, 9780099366515, 9780099537861, ...","[15741396, 85587, 895583, 11783478, 882524, 54...","[Arrow Books, Grand Central Publishing, Arrow,...",What's to Love: This gripping graphic novel ad...,"[19161909, 776159, 280727, 712372, 867622, 288...",1553,62702,3.75,397.0,0,"[{'count': '1088', 'name': 'mystery'}, {'count..."
2,995,Derailed,[57057],[],"[0446514152, 044661372X, 0446696692]","[9780446514156, 9780446613729, 9780446696692]","[5952238, 1387774, 314361]",[Grand Central Publishing],Two Strangers on a Train. A Taste of Temptatio...,"[130391, 596239, 1024724, 79727, 6431264, 2857...",10,37,3.77,418.0,0,"[{'count': '86', 'name': 'thriller'}, {'count'..."
3,1285,Plum Boxed Set 2,[2384],[183284],[0312947445],[9780312947446],[148140],[St. Martin's Paperbacks],This boxed set contains three hilarious Stepha...,"[287573, 12961168, 2137350, 1376643, 2860451, ...",30,1280,4.43,NaN,1,"[{'count': '37', 'name': 'mystery'}, {'count':..."
4,1616,The High Window,[1377],[855278],"[0140008519, 0241956293, 0140108939, 039475826...","[9780140008517, 9780241956298, 9780140108934, ...","[1930777, 25638586, 246991, 11373619, 2049, 74...","[Penguin, Ballantine Books, Penguin Books, Vin...","Raymond Chandler is a master."" --""The New York...","[30003, 867622, 280727, 844793, 19161909]",439,11370,4.08,272.0,0,"[{'count': '612', 'name': 'mystery'}, {'count'..."


In [145]:
df.shape

(63007, 16)

In [146]:
df.to_csv('../data/intermediate/books-cleaned/mystery.csv', index=False)